In [ ]:
import os
import json
import torch
import logging
from tqdm import tqdm
from transformers import AutoTokenizer
from peft import PeftModel
from unsloth import FastLanguageModel

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# 设置日志配置
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='inference_unsloth.log',
    filemode='a',
    encoding='utf-8'
)

logger = logging.getLogger(__name__)

In [ ]:
# 配置参数
MODEL_BASE_PATH = "../model/Qwen2.5-7B-Instruct"  # 基础模型路径
LORA_PATH = "../output/Qwen2.5-7B-Instruct-news-finetune-lora"  # LoRA权重路径
TEST_DATA_PATH = "../data/test_results.json"  # 测试集路径
OUTPUT_PATH = "../data/test_results.json"  # 输出结果路径
MAX_LENGTH = 2048  # 最大输入长度
MAX_NEW_TOKENS = 1024  # 最大生成的新token数量
BATCH_SIZE = 1  # 推理批次大小

In [ ]:
def load_model():
    """使用Unsloth加载模型和分词器"""
    logger.info("正在使用Unsloth加载模型和分词器...")
    
    # 使用Unsloth加载基础模型
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_BASE_PATH,
        max_seq_length=MAX_LENGTH,
        dtype=torch.bfloat16,
        load_in_4bit=True,
    )

    
    # # 加载LoRA权重
    # logger.info(f"正在加载LoRA权重: {LORA_PATH}")
    # model = PeftModel.from_pretrained(model, LORA_PATH)
    
    # 合并LoRA权重（可选，如果内存允许的话）
    # logger.info("合并LoRA权重到基础模型...")
    # model = model.merge_and_unload()
    
    model.eval()  # 设置为评估模式
    
    return model, tokenizer


In [ ]:
def format_prompt(question):
    """格式化提问"""
    # 使用与训练时相同的格式
    messages = [
        {"role": "user", "content": f"回答以下问题：{question}"}
    ]
    return messages

In [ ]:
def generate_answer(model, tokenizer, question):
    """使用Unsloth生成回答"""
    messages = format_prompt(question)
    prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,  # 关键！仅格式化不编码
            add_generation_prompt=True  # 为assistant添加空内容占位符
        )
    
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()} 
    
    # 使用模型生成回答
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,  # 使用贪婪解码
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # 解码生成的回答
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # 提取助手的回答部分
    assistant_start = generated_text.rfind("<|im_start|>assistant\n")
    if assistant_start != -1:
        # 截取助手部分的回答
        assistant_text = generated_text[assistant_start + len("<|im_start|>assistant\n"):]
        # 去除结束标记
        end_token_pos = assistant_text.find("<|im_end|>")
        if end_token_pos != -1:
            assistant_text = assistant_text[:end_token_pos]
        return assistant_text.strip()
    else:
        # 如果找不到标记，直接返回生成的文本
        return generated_text.replace(prompt, "").strip()

In [ ]:
# 加载模型和分词器
model, tokenizer = load_model()

In [ ]:
# 加载测试数据
logger.info(f"正在加载测试数据: {TEST_DATA_PATH}")
with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

# 初始化结果列表
results = []

In [ ]:
# 对每个测试样本进行推理
logger.info(f"开始对{len(test_data)}条测试数据进行推理...")
for item in tqdm(test_data):
    question = item["question"]
    
    # 生成回答
    base_model_answer = generate_answer(model, tokenizer, question)
    
    # 创建结果字典
    result = {
        "question": item["question"],
        "answer": item["answer"],
        "model_answer": item["model_answer"],
        "base_model_answer": base_model_answer,
        "source_id": item.get("sourse_id", item.get("source_id", 0)),  # 处理可能的拼写错误
        "type": item.get("type", "")
    }
    
    results.append(result)



In [ ]:
question = "姜景峰在2016年1月7日走访上海飞科电器股份有限公司时，主要向在沪永商传达了哪些信息？"
model_answer = generate_answer(model, tokenizer, question)

In [ ]:
print(model_answer)

In [ ]:
# 保存结果
logger.info(f"推理完成，正在保存结果到: {OUTPUT_PATH}")
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

logger.info("所有操作已完成")

In [ ]:
model2, tokenizer = FastLanguageModel.from_pretrained(
        model_name="../model/Qwen2.5-7B-Instruct",
        max_seq_length=MAX_LENGTH,
        dtype=torch.bfloat16,
        load_in_4bit=True,
    )

In [ ]:
question ="2016年猴年贺岁普通纪念币的第一批次预约时间是什么时候？"


In [ ]:
# 生成回答
model_answer = generate_answer(model, tokenizer, question)
print(model_answer)

In [ ]:
model2_answer = generate_answer(model2, tokenizer, question)
print(model2_answer)